# GraphNull null ladder (CPU, internet on)

Clones click, flask and attrs at pinned commits, builds a static call graph before each of 400 recent commits per repository, and asks how often the neighbour list of one edited function contains another function edited in the same commit. It compares the real graph with three null graphs from GraphNull: uniform random with the same edge count, degree-preserving rewire, and degree-preserving rewire that also keeps every node's count of neighbours per source file. No competition data is read. Apache 2.0.

In [ ]:
import os
os.makedirs('/kaggle/working/graphnull_src', exist_ok=True)
os.chdir('/kaggle/working/graphnull_src')
open('graphnull.py', 'w').write('# Copyright 2026 Yerramsetty Sai Venkata Suchita, Manjunath Mailey, Kanuma Jagan Mohan.\n# Licensed under the Apache License, Version 2.0.\n"""GraphNull: interface-matched null worlds for code-graph tools.\n\nA null world keeps everything an agent can see about a graph tool (tool names,\nnode set, output format, every node\'s degree) and scrambles only the content\nthat a structural claim depends on. If an agent does as well in the null world\nas in the real one, the gain cannot be credited to the structure.\n\nNull ladder, from weakest to strongest:\n  random      same node set and edge count, edges placed uniformly (degrees change)\n  degree      degree-preserving double-edge swaps (every node keeps its degree)\n  stratified  degree-preserving swaps that also keep, for every node, how many of\n              its neighbours sit in each group (for example each source file)\n  permuted    real graph, embedding vectors permuted across nodes\n\nOnly the standard library and numpy are needed. Every function is\ndeterministic given its seed. The harness file format is a JSON graph with\n"nodes" and "edges" (source, target, type) and an .npz embedding file keyed by\nnode id, as in the Gemma 4 Developer Agent competition.\n"""\nimport collections\nimport glob\nimport hashlib\nimport json\nimport os\nimport random\n\nimport numpy as np\n\n__version__ = "1.0.0"\nDEFAULT_SEED = 20261108\nSWAPS_PER_EDGE = 10\n\n\n# ---------------------------------------------------------------- seeds and io\ndef name_seed(name, seed=DEFAULT_SEED):\n    """Stable per-object seed derived from a global seed and a name."""\n    return int(hashlib.sha256(f"{seed}:{name}".encode()).hexdigest()[:12], 16)\n\n\ndef normalize(edges):\n    """Edge dicts or (u, v) tuples to a set of undirected pairs and a set of self loops."""\n    pairs, loops = set(), set()\n    for e in edges:\n        u, v = (e["source"], e["target"]) if isinstance(e, dict) else e\n        if u == v:\n            loops.add(u)\n        else:\n            pairs.add((u, v) if u < v else (v, u))\n    return pairs, loops\n\n\ndef degrees(pairs):\n    d = collections.Counter()\n    for u, v in pairs:\n        d[u] += 1\n        d[v] += 1\n    return d\n\n\ndef adjacency(pairs):\n    a = collections.defaultdict(set)\n    for u, v in pairs:\n        a[u].add(v)\n        a[v].add(u)\n    return a\n\n\ndef emit(graph, pairs, loops, edge_type="calls"):\n    """Write pairs back in the harness format: both directions, one edge type."""\n    edges = []\n    for u, v in sorted(pairs):\n        edges.append({"source": u, "target": v, "type": edge_type})\n        edges.append({"source": v, "target": u, "type": edge_type})\n    for u in sorted(loops):\n        edges.append({"source": u, "target": u, "type": edge_type})\n    out = {k: v for k, v in graph.items() if k not in ("edges", "links")}\n    out["edges"] = edges\n    return out\n\n\ndef write_json(obj, path):\n    blob = json.dumps(obj, sort_keys=True).encode()\n    with open(path, "wb") as f:\n        f.write(blob)\n    return hashlib.sha256(blob).hexdigest()\n\n\ndef file_hash(path):\n    with open(path, "rb") as f:\n        return hashlib.sha256(f.read()).hexdigest()\n\n\n# ---------------------------------------------------------------- null models\ndef rewire_degree(pairs, seed, swaps_per_edge=SWAPS_PER_EDGE, accept=None):\n    """Degree-preserving double-edge swaps.\n\n    Picks two edges (a, b) and (c, d) and replaces them with (a, d) and (c, b)\n    unless that creates a self loop or a duplicate edge. `accept(a, b, c, d)`\n    can veto a swap (used by the stratified null). Returns (new pair set, share of the\n    requested swaps that were completed).\n    """\n    rng = random.Random(seed)\n    E = sorted(pairs)\n    S = set(E)\n    m = len(E)\n    done, tries = 0, 0\n    while done < swaps_per_edge * m and tries < swaps_per_edge * m * 50:\n        tries += 1\n        i = int(rng.random() * m)\n        j = int(rng.random() * m)\n        if i == j:\n            continue\n        a, b = E[i]\n        c, d = E[j]\n        if rng.random() < 0.5:\n            c, d = d, c\n        if len({a, b, c, d}) < 4:\n            continue\n        if accept is not None and not accept(a, b, c, d):\n            continue\n        e1 = (a, d) if a < d else (d, a)\n        e2 = (c, b) if c < b else (b, c)\n        if e1 in S or e2 in S:\n            continue\n        S.discard(E[i])\n        S.discard(E[j])\n        S.add(e1)\n        S.add(e2)\n        E[i], E[j] = e1, e2\n        done += 1\n    return set(E), done / max(swaps_per_edge * m, 1)\n\n\ndef rewire_stratified(pairs, group_of, seed, swaps_per_edge=SWAPS_PER_EDGE):\n    """Degree-preserving swaps restricted to partners in the same group.\n\n    A swap (a, b), (c, d) -> (a, d), (c, b) is allowed only when b and d are in\n    the same group and a and c are in the same group. Then each of the four\n    endpoints gives up one neighbour and receives another from the same group,\n    so every node keeps both its degree and its count of neighbours per group. With group = source file this\n    keeps file locality and scrambles which function in the file is called.\n    """\n    return rewire_degree(pairs, seed, swaps_per_edge,\n                         accept=lambda a, b, c, d: group_of(b) == group_of(d) and group_of(a) == group_of(c))\n\n\ndef random_same_size(nodes, n_edges, seed):\n    """Uniform random simple graph on the same nodes with the same edge count."""\n    rng = random.Random(seed)\n    nodes = sorted(nodes)\n    out = set()\n    if len(nodes) < 2:\n        return out\n    cap = len(nodes) * (len(nodes) - 1) // 2\n    n_edges = min(n_edges, cap)\n    while len(out) < n_edges:\n        u, v = rng.sample(nodes, 2)\n        out.add((u, v) if u < v else (v, u))\n    return out\n\n\ndef permute_embeddings(keys, matrix, seed):\n    """Same keys, same vectors, assigned to different nodes."""\n    perm = np.random.default_rng(seed).permutation(len(keys))\n    return matrix[perm]\n\n\n# ---------------------------------------------------------------- checks\ndef group_profile(pairs, group_of):\n    """For every node, a Counter of neighbour groups."""\n    prof = collections.defaultdict(collections.Counter)\n    for u, v in pairs:\n        prof[u][group_of(v)] += 1\n        prof[v][group_of(u)] += 1\n    return prof\n\n\ndef check_null(real, null, group_of=None):\n    """Invariants of a null graph against the real one."""\n    out = {\n        "same_edge_count": len(real) == len(null),\n        "degree_preserved": degrees(real) == degrees(null),\n        "edges_changed": 1 - len(real & null) / max(len(real), 1),\n    }\n    if group_of is not None:\n        out["group_profile_preserved"] = group_profile(real, group_of) == group_profile(null, group_of)\n    return out\n\n\n# ---------------------------------------------------------------- harness worlds\ndef make_worlds(graph_dir, emb_dir, out_dir, seed=DEFAULT_SEED, swaps_per_edge=SWAPS_PER_EDGE,\n                stratified_group=None, verbose=True):\n    """Build harness-compatible worlds from a directory of graphs and embeddings.\n\n    W1 real graph (normalized), W2 degree-preserving rewire, W3 real graph with\n    permuted embeddings. If `stratified_group` is a function node_id -> group,\n    W2S (degree and group preserving) is also written. Graph and embedding files\n    keep their names, so a harness pointed at out_dir/W2 cannot tell the worlds\n    apart by interface. Returns (manifest dict, manifest sha256 prefix, report).\n    """\n    gfiles = {os.path.basename(p)[:-5]: p for p in glob.glob(os.path.join(graph_dir, "*.json"))}\n    efiles = {os.path.basename(p)[:-4]: p for p in glob.glob(os.path.join(emb_dir, "*.npz"))}\n    worlds = ["W1", "W2", "W3"] + (["W2S"] if stratified_group else [])\n    for w in worlds:\n        for sub in ("graphs", "embeddings"):\n            os.makedirs(os.path.join(out_dir, w, sub), exist_ok=True)\n    manifest = {"seed": seed, "files": {}}\n    report = collections.defaultdict(list)\n    for k, name in enumerate(sorted(gfiles)):\n        with open(gfiles[name]) as f:\n            G = json.load(f)\n        pairs, loops = normalize(G["edges"])\n        s = name_seed(name, seed)\n        rpairs, frac = rewire_degree(pairs, s, swaps_per_edge)\n        chk = check_null(pairs, rpairs)\n        report["swaps_completed"].append(frac)\n        report["degree_preserved"].append(chk["degree_preserved"])\n        report["edges_changed"].append(chk["edges_changed"])\n        g1 = os.path.join(out_dir, "W1", "graphs", name + ".json")\n        g2 = os.path.join(out_dir, "W2", "graphs", name + ".json")\n        g3 = os.path.join(out_dir, "W3", "graphs", name + ".json")\n        h1 = write_json(emit(G, pairs, loops), g1)\n        h2 = write_json(emit(G, rpairs, loops), g2)\n        if not os.path.lexists(g3):\n            os.symlink(os.path.abspath(g1), g3)\n        entry = {"W1_graph": h1, "W2_graph": h2}\n        if stratified_group:\n            spairs, _ = rewire_stratified(pairs, stratified_group, s, swaps_per_edge)\n            sc = check_null(pairs, spairs, stratified_group)\n            report["stratified_profile_preserved"].append(sc["group_profile_preserved"] and sc["degree_preserved"])\n            report["stratified_edges_changed"].append(sc["edges_changed"])\n            gs = os.path.join(out_dir, "W2S", "graphs", name + ".json")\n            entry["W2S_graph"] = write_json(emit(G, spairs, loops), gs)\n        if name in efiles:\n            src = efiles[name]\n            for w in ["W1", "W2"] + (["W2S"] if stratified_group else []):\n                dst = os.path.join(out_dir, w, "embeddings", name + ".npz")\n                if not os.path.lexists(dst):\n                    os.symlink(os.path.abspath(src), dst)\n            E = np.load(src)\n            keys = list(E.keys())\n            M = np.stack([E[x] for x in keys])\n            Mp = permute_embeddings(keys, M, s)\n            dst3 = os.path.join(out_dir, "W3", "embeddings", name + ".npz")\n            np.savez(dst3, **{x: Mp[i] for i, x in enumerate(keys)})\n            E3 = np.load(dst3)\n            report["embeddings_same_keys"].append(sorted(E3.keys()) == sorted(keys) and E3[keys[0]].dtype == M.dtype)\n            entry["emb"] = file_hash(src)\n            entry["W3_emb"] = file_hash(dst3)\n        if stratified_group:\n            manifest["files"][name] = entry\n        else:\n            manifest["files"][name] = {key: entry[key] for key in ("W1_graph", "W2_graph", "emb", "W3_emb") if key in entry}\n        if verbose and k % 25 == 0:\n            print(f"  graph {k + 1}/{len(gfiles)}")\n    blob = json.dumps(manifest, sort_keys=True).encode()\n    with open(os.path.join(out_dir, "manifest.json"), "wb") as f:\n        f.write(blob)\n    return manifest, hashlib.sha256(blob).hexdigest()[:16], dict(report)\n\n\ndef summarize(report):\n    """One line per invariant: count of passes and min / median of shares."""\n    lines = []\n    for key, vals in report.items():\n        if vals and isinstance(vals[0], bool):\n            lines.append(f"{key}: {sum(vals)} / {len(vals)}")\n        elif vals:\n            lines.append(f"{key}: min {min(vals):.3f} | median {float(np.median(vals)):.3f}")\n    return "\\n".join(lines)\n\n\nif __name__ == "__main__":\n    import argparse\n    ap = argparse.ArgumentParser(description="Build GraphNull worlds for a harness graph directory.")\n    ap.add_argument("graph_dir")\n    ap.add_argument("emb_dir")\n    ap.add_argument("out_dir")\n    ap.add_argument("--seed", type=int, default=DEFAULT_SEED)\n    ap.add_argument("--swaps-per-edge", type=int, default=SWAPS_PER_EDGE)\n    ap.add_argument("--stratify-by-file", action="store_true",\n                    help="also write W2S, grouping nodes by the module part of their id")\n    a = ap.parse_args()\n    grp = (lambda n: n.rsplit(".", 1)[0]) if a.stratify_by_file else None\n    _, h, rep = make_worlds(a.graph_dir, a.emb_dir, a.out_dir, a.seed, a.swaps_per_edge, grp)\n    print(summarize(rep))\n    print("manifest sha256:", h)\n')
print('wrote', 'graphnull.py', len(open('graphnull.py').read().splitlines()), 'lines')
open('heldout_core.py', 'w').write('# Copyright 2026 InnovativeOrbits. Licensed under the Apache License, Version 2.0.\n# GraphNull held-out check. CPU only. Uses three public repositories that are\n# not part of the Gemma 4 Developer Agent competition data. No competition\n# data is read or written by this notebook.\nimport ast, collections, hashlib, io, json, os, random, re, subprocess, tarfile, time\nimport numpy as np\n\nON_KAGGLE = os.path.isdir("/kaggle/working")\nWORK = "/tmp/heldout" if ON_KAGGLE else os.environ.get("HELDOUT_WORK", "./heldout_work")\nRESULTS = "/kaggle/working/heldout_results.json" if ON_KAGGLE else "heldout_results.json"\nREPOS = [  # (name, url, pinned commit)\n    ("click", "https://github.com/pallets/click.git", "8abb77a8432e03ed14a95f972b44f905279415a7"),\n    ("flask", "https://github.com/pallets/flask.git", "49943b5a94413a530761350f04e896f76e8d620e"),\n    ("attrs", "https://github.com/python-attrs/attrs.git", "9d9f94a043de7fc4012459853a4ed597a5577855"),\n]\nSEED = 20261108        # same seed as the main study\nN_COMMITS = 400        # most recent non-merge commits scanned per repository\nN_NULLS = 20           # rewired graphs per task\nN_BOOT = 10000\nos.makedirs(WORK, exist_ok=True)\n\ndef git(repo, *args, binary=False):\n    out = subprocess.run(["git", "-C", repo, *args], capture_output=True, check=True).stdout\n    return out if binary else out.decode("utf-8", "replace")\n\n# ---------------- 1. call graph builder (static, approximate) ----------------\nSKIP_DIRS = ("tests/", "test/", "docs/", "examples/", "benchmarks/", "bench/")\nBUILTIN_METHODS = set()\nfor _t in (list, dict, str, set, tuple, bytes, int, float, object):\n    BUILTIN_METHODS |= set(dir(_t))\n\ndef is_source(path):\n    p = path.lower()\n    if not p.endswith(".py"):\n        return False\n    if any(("/" + p).find("/" + d) >= 0 for d in SKIP_DIRS):\n        return False\n    base = p.split("/")[-1]\n    return not (base.startswith("test_") or base.endswith("_test.py")\n                or base in ("conftest.py", "setup.py", "noxfile.py"))\n\ndef module_of(path):\n    m = path[:-3].replace("/", ".")\n    return m[4:] if m.startswith("src.") else m\n\nDEFS = (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)\n\ndef index_file(path, text):\n    """Return list of (node_id, kind, first_line, last_line, ast_node, class_id)."""\n    try:\n        tree = ast.parse(text)\n    except (SyntaxError, ValueError):\n        return []\n    out = []\n    def walk(node, prefix, cls):\n        for ch in ast.iter_child_nodes(node):\n            if isinstance(ch, DEFS):\n                nid = prefix + "." + ch.name\n                kind = "class" if isinstance(ch, ast.ClassDef) else "function"\n                first = min([ch.lineno] + [d.lineno for d in ch.decorator_list])\n                out.append((nid, kind, first, ch.end_lineno, ch, cls))\n                walk(ch, nid, nid if kind == "class" else cls)\n            else:\n                walk(ch, prefix, cls)\n    walk(tree, module_of(path), None)\n    return out\n\ndef own_calls(fn):\n    """Names called directly inside fn, not inside nested defs."""\n    calls = []\n    stack = list(ast.iter_child_nodes(fn))\n    while stack:\n        n = stack.pop()\n        if isinstance(n, DEFS):\n            continue\n        if isinstance(n, ast.Call):\n            f = n.func\n            if isinstance(f, ast.Name):\n                calls.append(("name", f.id))\n            elif isinstance(f, ast.Attribute):\n                on_self = isinstance(f.value, ast.Name) and f.value.id in ("self", "cls")\n                calls.append(("self" if on_self else "attr", f.attr))\n        stack.extend(ast.iter_child_nodes(n))\n    return calls\n\ndef build_graph(files):\n    """files: dict path -> source text. Returns (node ids, undirected pair set, spans)."""\n    entries, spans = [], {}\n    for path in sorted(files):\n        rows = index_file(path, files[path])\n        spans[path] = [(a, b, nid) for nid, _, a, b, _, _ in rows]\n        entries += [(path, r) for r in rows]\n    by_short = collections.defaultdict(list)\n    for path, (nid, kind, a, b, node, cls) in entries:\n        by_short[nid.split(".")[-1]].append((nid, module_of(path)))\n    ids = sorted({r[0] for _, r in entries})\n    idset = set(ids)\n    pairs = set()\n    for path, (nid, kind, a, b, node, cls) in entries:\n        if kind != "function":\n            continue\n        mod = module_of(path)\n        for how, name in own_calls(node):\n            target = None\n            if how == "self" and cls and (cls + "." + name) in idset:\n                target = cls + "." + name\n            else:\n                cands = by_short.get(name, [])\n                same = [c for c, m in cands if m == mod]\n                if how == "name":\n                    if len(same) == 1:\n                        target = same[0]\n                    elif not same and len(cands) == 1:\n                        target = cands[0][0]\n                elif len(name) >= 4 and name not in BUILTIN_METHODS and len(cands) == 1:\n                    target = cands[0][0]\n            if target and target != nid:\n                pairs.add((nid, target) if nid < target else (target, nid))\n    return ids, pairs, spans\n\n# ---------------- 2. GraphNull rewiring (identical to the main study) ----------------\ndef name_seed(name):\n    h = hashlib.sha256(f"{SEED}:{name}".encode()).hexdigest()\n    return int(h[:12], 16)\n\ndef swap(pairs, seed, mult=10):\n    rng = random.Random(seed)\n    E = sorted(pairs)\n    S = set(E)\n    m = len(E)\n    done, tries = 0, 0\n    while done < mult * m and tries < mult * m * 50:\n        tries += 1\n        i = int(rng.random() * m)\n        j = int(rng.random() * m)\n        if i == j:\n            continue\n        a, b = E[i]\n        c, d = E[j]\n        if rng.random() < 0.5:\n            c, d = d, c\n        if len({a, b, c, d}) < 4:\n            continue\n        e1 = (a, d) if a < d else (d, a)\n        e2 = (c, b) if c < b else (b, c)\n        if e1 in S or e2 in S:\n            continue\n        S.discard(E[i])\n        S.discard(E[j])\n        S.add(e1)\n        S.add(e2)\n        E[i], E[j] = e1, e2\n        done += 1\n    return set(E), done / max(mult * m, 1)\n\ndef degrees(pairs):\n    d = collections.Counter()\n    for u, v in pairs:\n        d[u] += 1\n        d[v] += 1\n    return d\n\ndef adjacency(pairs):\n    a = collections.defaultdict(set)\n    for u, v in pairs:\n        a[u].add(v)\n        a[v].add(u)\n    return a\n\n# ---------------- 3. tasks from commit history ----------------\nHUNK = re.compile(r"@@ -(\\d+)(?:,(\\d+))? \\+(\\d+)(?:,(\\d+))? @@")\n\ndef changed_old_lines(repo, parent, commit):\n    """dict path -> set of old-side line numbers touched (modified files only)."""\n    diff = git(repo, "diff", "-U0", "--no-renames", "--diff-filter=M", parent, commit, "--", "*.py")\n    out, cur = collections.defaultdict(set), None\n    for line in diff.splitlines():\n        if line.startswith("--- a/"):\n            cur = line[6:]\n        elif line.startswith("@@") and cur:\n            m = HUNK.match(line)\n            start, n = int(m.group(1)), int(m.group(2) or 1)\n            out[cur].update(range(start, start + n) if n else [start])\n    return out\n\ndef source_at(repo, commit):\n    blob = git(repo, "archive", "--format=tar", commit, binary=True)\n    files = {}\n    with tarfile.open(fileobj=io.BytesIO(blob)) as tf:\n        for m in tf.getmembers():\n            if m.isfile() and is_source(m.name):\n                files[m.name] = tf.extractfile(m).read().decode("utf-8", "replace")\n    return files\n\ndef gold_nodes(spans, touched):\n    gold = set()\n    for path, lines in touched.items():\n        for ln in lines:\n            inner = None\n            for a, b, nid in spans.get(path, ()):\n                if a <= ln <= b and (inner is None or b - a < inner[0]):\n                    inner = (b - a, nid)\n            if inner:\n                gold.add(inner[1])\n    return gold\n\ndef recall(adj, gold, hops):\n    """Share of edited functions that have another edited function within `hops`."""\n    hit = 0\n    for g in gold:\n        seen, frontier = {g}, {g}\n        for _ in range(hops):\n            frontier = {v for u in frontier for v in adj.get(u, ())} - seen\n            seen |= frontier\n        hit += bool((seen - {g}) & gold)\n    return hit / len(gold)\n\ndef boot_ci(x, n=N_BOOT, seed=SEED):\n    x = np.asarray(x, float)\n    rng = np.random.default_rng(seed)\n    means = x[rng.integers(0, len(x), size=(n, len(x)))].mean(axis=1)\n    return float(x.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))\n\n')
print('wrote', 'heldout_core.py', len(open('heldout_core.py').read().splitlines()), 'lines')
open('ladder.py', 'w').write('# Copyright 2026 Yerramsetty Sai Venkata Suchita, Manjunath Mailey, Kanuma Jagan Mohan.\n# Licensed under the Apache License, Version 2.0.\n# GraphNull null ladder on three public repositories (click, flask, attrs).\n# CPU only, internet on (clones the repositories). Reads no competition data.\n# For each recent commit that edits 2 to 10 functions, asks how often the\n# neighbour list of one edited function contains another edited function,\n# in the real call graph and in three null graphs of increasing strength.\nimport collections, hashlib, json, os, sys, time\nimport numpy as np\nHERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else "."\nsys.path.insert(0, HERE)\nimport graphnull as gn\n\nsrc = open(os.path.join(HERE, "heldout_core.py")).read()\nexec(src)  # REPOS, SEED, git, is_source, module_of, index_file, build_graph, changed_old_lines, source_at, gold_nodes, recall, boot_ci, WORK\n\nN_NULLS = int(os.environ.get("N_NULLS", "20"))\nLEVELS = ["random", "degree", "stratified"]\nt0 = time.time()\nrows, inv = [], collections.defaultdict(list)\nfor name, url, pin in REPOS:\n    repo = f"{WORK}/{name}"\n    if not os.path.isdir(repo):\n        subprocess.run(["git", "clone", "-q", url, repo], check=True)\n    commits = git(repo, "log", "--no-merges", "--format=%H %P", "-n", str(N_COMMITS), pin, "--", "*.py").split("\\n")\n    kept = 0\n    for line in commits:\n        parts = line.split()\n        if len(parts) != 2:\n            continue\n        commit, parent = parts\n        touched = {p: l for p, l in changed_old_lines(repo, parent, commit).items() if is_source(p)}\n        if not touched or len(touched) > 8:\n            continue\n        ids, pairs, spans = build_graph(source_at(repo, parent))\n        gold = gold_nodes(spans, touched)\n        if not (2 <= len(gold) <= 10) or len(pairs) < 50:\n            continue\n        grp = {nid: module_of(path) for path, sp in spans.items() for _, _, nid in sp}\n        group_of = lambda n: grp.get(n, "?")\n        adj1 = gn.adjacency(pairs)\n        row = {"repo": name, "commit": commit[:12], "nodes": len(ids), "edges": len(pairs), "gold": len(gold),\n               "same_file_share": float(np.mean([group_of(u) == group_of(v) for u, v in pairs]))}\n        for h in (1, 2):\n            row[f"real_{h}"] = recall(adj1, gold, h)\n        acc = {(lv, h): [] for lv in LEVELS for h in (1, 2)}\n        for k in range(N_NULLS):\n            s = gn.name_seed(f"{name}:{commit}:{k}", SEED)\n            nulls = {\n                "random": gn.random_same_size(ids, len(pairs), s),\n                "degree": gn.rewire_degree(pairs, s)[0],\n                "stratified": gn.rewire_stratified(pairs, group_of, s)[0],\n            }\n            for lv, npairs in nulls.items():\n                c = gn.check_null(pairs, npairs, group_of)\n                for key, v in c.items():\n                    inv[(lv, key)].append(v)\n                a2 = gn.adjacency(npairs)\n                for h in (1, 2):\n                    acc[(lv, h)].append(recall(a2, gold, h))\n        for (lv, h), v in acc.items():\n            row[f"{lv}_{h}"] = float(np.mean(v))\n        rows.append(row)\n        kept += 1\n    print(f"{name}: kept {kept} tasks | {round(time.time() - t0)} s", flush=True)\n\ndef fmt(t):\n    return f"{100 * t[0]:5.1f} [{100 * t[1]:5.1f}, {100 * t[2]:5.1f}]"\n\nprint("\\n=== NULL INVARIANTS (all repositories, all null graphs) ===")\nfor lv in LEVELS:\n    d = {k[1]: v for k, v in inv.items() if k[0] == lv}\n    print(f"{lv:10s} same edge count {sum(d[\'same_edge_count\'])}/{len(d[\'same_edge_count\'])}"\n          f" | degree kept {sum(d[\'degree_preserved\'])}/{len(d[\'degree_preserved\'])}"\n          f" | file profile kept {sum(d[\'group_profile_preserved\'])}/{len(d[\'group_profile_preserved\'])}"\n          f" | edges changed median {np.median(d[\'edges_changed\']):.3f} min {min(d[\'edges_changed\']):.3f}")\n\nprint(f"\\n=== NULL LADDER: co-edit recall (%), {len(rows)} commits, task bootstrap 95% CI ===")\nprint("share of real call edges inside one file, median:", round(float(np.median([r[\'same_file_share\'] for r in rows])), 3))\nfor h in (1, 2):\n    print(f"\\n{h}-hop")\n    for lv in ["real"] + LEVELS[::-1]:\n        print(f"  {lv:10s} {fmt(boot_ci([r[f\'{lv}_{h}\'] for r in rows]))}")\n    for lv in LEVELS[::-1]:\n        print(f"  real - {lv:10s} {fmt(boot_ci([r[f\'real_{h}\'] - r[f\'{lv}_{h}\'] for r in rows]))}")\n    print(f"  stratified - degree  {fmt(boot_ci([r[f\'stratified_{h}\'] - r[f\'degree_{h}\'] for r in rows]))}")\n    print(f"  degree - random      {fmt(boot_ci([r[f\'degree_{h}\'] - r[f\'random_{h}\'] for r in rows]))}")\nfor name, _, _ in REPOS:\n    rs = [r for r in rows if r["repo"] == name]\n    print(f"{name:6s} n={len(rs):3d} 1-hop " + "  ".join(f"{lv}={100*np.mean([r[f\'{lv}_1\'] for r in rs]):.1f}" for lv in ["real"] + LEVELS[::-1]))\n\nblob = json.dumps({"seed": SEED, "n_nulls": N_NULLS, "repos": REPOS, "rows": rows}, sort_keys=True).encode()\nout = "/kaggle/working/ladder_results.json" if os.path.isdir("/kaggle/working") else "ladder_results.json"\nopen(out, "wb").write(blob)\nprint("\\nresults sha256:", hashlib.sha256(blob).hexdigest()[:16])\nprint("graphnull version:", gn.__version__, "| total seconds:", round(time.time() - t0))\nprint("LADDER DONE")\n')
print('wrote', 'ladder.py', len(open('ladder.py').read().splitlines()), 'lines')

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, 'ladder.py'], capture_output=True, text=True)
print(r.stdout[-6000:])
print(r.stderr[-3000:])